In [1]:
#Adapted: https://github.com/VelardeGissel/AutomaticMusicOrchestration/blob/main/notebooks/amo_with_doublings_multiclass_demo.ipynb
#Amo demo
#GV 19.9.2025 + FM 22/10/2025 + FM 05.11.2025 + GV 29.1.2026
import sys
sys.path.extend(['src', '../src']) 
# Import all ML orchestration functions
from amo.ml_orchestration import (
    ml_exp_with_timing,  # Main experiment function with timing preservation
    extract_timing_structure,
    save_midi_with_exact_timing_structure,
    KerasClassifierWrapper,
    build_lstm_classifier,
    build_transformer_classifier,
    clf_predict,
    defineXy,           # Data preparation function
    split_and_encode    # Train/test split with encoding
)
import os
# Import MIDI processing functions  
from amo.midi2df2midi import midi_to_dataframe, save_midi_from_df
from amo.mappings import fill_quaterna_columns, learn_quaterna_mapping
# Import amo funtion
from amo.ml_orchestration import amo_with_doublings_multiclass

import pandas as pd
import numpy as np
from collections import defaultdict

In [2]:
#get the parent directory
cwd = os.getcwd()
parent = os.path.normpath(os.path.join(cwd, '..', '..'))
path_AMO = parent + '/AutomaticMusicOrchestration'
model_path = path_AMO + '/'
#print(parent)#.../test
#print(path_AMO)

In [3]:
def transpose(note, inverse=False, n_semitones=12):
    # Example: transpose pitch
    if inverse:
        n_semitones = - n_semitones
    new_note = note.copy()
    new_note['pitch'] = note['pitch'] + n_semitones
    return [ new_note ]

In [4]:
transformations = [
    (transpose, {'n_semitones': 12}),
    (transpose, {'n_semitones': 24}),
    (transpose, {'n_semitones': 46}),
    (transpose, {'n_semitones': -12}),
    (transpose, {'n_semitones': -24}),
    (transpose, {'n_semitones': -46}),
    #(split_duration, {'smallest_unit': 0.25}),
]

In [5]:
model="XGBoost"
#model="AdaBoost"

In [6]:
# Specify the file path
file_orch=parent+'/music/LOP_database_06_09_17/hand_picked_Spotify/39/swan_lake_09_orch.mid'
file_solo=path_AMO+'/data/samples/midis/Autumn.mid'
filein=file_orch
fileout=file_solo
# Check if the file exists
if os.path.exists(file_solo) and os.path.exists(file_solo):
   print("Both files exist")
else:
   print("Check if the files exist.")

Both files exist


In [7]:
# filein='data/samples/midis/symphony_7_1_orch.mid'
#filein='LOP_database_06_09_17/hand_picked_Spotify/39/swan_lake_09_orch.mid'
#fileout='AutomaticMusicOrchestration/midis/fur-elise.mid'
# Multiclass, with transformations
amo_with_doublings_multiclass(file_orch,file_solo,model=model, tol=0.2, transformations=transformations, model_path=model_path)


========= PREPROCESSING =========
[PREPROCESSING] Computing and caching results...
{'transformed': 284, 'direct': 41, 'none': 6029}
Original size: (6354, 15)
Dropping [2514, 2517, 2520, 2522, 2525, 2529, 2532, 2535, 2538, 2542, 2552, 2987, 333, 2989, 2991, 2993, 2995, 2997, 2999, 3001, 3003, 3005, 3007, 3009, 3011, 3013, 3015, 3017, 334, 335, 336, 337, 338, 339, 340, 341, 342, 343, 344, 345, 346, 347, 348, 349, 350, 351, 352, 353, 354, 355, 356, 357, 358, 359, 360, 361, 362, 363, 364, 365, 366, 367, 4629, 51, 4631, 53, 4632, 369, 55, 4633, 57, 4634, 59, 4635, 61, 4636, 63, 4637, 65, 4638, 67, 4639, 69, 4640, 71, 4641, 73, 4642, 75, 4643, 77, 4644, 79, 4645, 81, 4646, 549, 83, 4647, 85, 4648, 87, 4649, 89, 4650, 91, 4651, 551, 93, 4652, 552, 553, 95, 4653, 555, 97, 4654, 99, 4655, 101, 4656, 103, 4657, 105, 4658, 383, 107, 4659, 109, 4660, 111, 4661, 113, 4662, 115, 4663, 117, 4664, 119, 4665, 121, 4666, 123, 4667, 125, 4668, 127, 4669, 129, 4670, 131, 4671, 133, 4672, 558, 135, 4673, 

{'time': 2.626499891281128,
 'accuracy (test)': 0.9319455564451561,
 'accuracy (train)': 0.9705646776131358,
 'precision (test)': 0.9578025477707006,
 'precision (train)': 0.9903807615230461,
 'recall (test)': 0.9405785770132916,
 'recall (train)': 0.9737931034482759,
 'f1 (test)': 0.949112426035503,
 'f1 (train)': 0.9820168902136116,
 "time (transpose_{'n_semitones': 12})": 0.4386119842529297,
 "accuracy (test) (transpose_{'n_semitones': 12})": 0.9900497512437811,
 "accuracy (train) (transpose_{'n_semitones': 12})": 0.998133941530168,
 "precision (test) (transpose_{'n_semitones': 12})": 0.9900497512437811,
 "precision (train) (transpose_{'n_semitones': 12})": 0.9981732475299606,
 "recall (test) (transpose_{'n_semitones': 12})": 0.9900497512437811,
 "recall (train) (transpose_{'n_semitones': 12})": 0.998133941530168,
 "f1 (test) (transpose_{'n_semitones': 12})": 0.9900497512437811,
 "f1 (train) (transpose_{'n_semitones': 12})": 0.9981487738002142,
 "time (transpose_{'n_semitones': 24})

In [9]:
# Multiclass, no transformations
amo_with_doublings_multiclass(file_orch,file_solo,model=model, tol=0.2, transformations=None, model_path=model_path)


========= PREPROCESSING =========
[PREPROCESSING] Computing and caching results...
Mapping: [[1 'Piccolo' 1 72]
 [2 '2 Flauti' 1 73]
 [3 '2 Oboi' 2 68]
 [4 '2 Clarinetti in A' 3 71]
 [5 '2 Fagotti' 4 70]
 [6 'Corni in F I II' 5 60]
 [7 'Corni in F III IV' 5 60]
 [8 '2 Cornets à piston in A' 6 56]
 [9 '2 Trombe in F' 6 56]
 [10 '2 Tromboni' 7 57]
 [11 'Trombone 3' 7 57]
 [12 'Tuba' 8 58]
 [13 'Timpani' 10 47]
 [14 'Piatti' 9 0]
 [15 'Gran Cassa' 10 47]
 [16 'Arpa' 0 46]
 [17 'Violine I' 11 48]
 [18 'Violine II' 12 48]
 [19 'Viola' 13 48]
 [20 'Violoncelle' 14 45]
 [20 'Violoncelle' 14 48]
 [21 'Contrabasse' 15 45]
 [21 'Contrabasse' 15 48]]

Creating multi-hot encoding for notes with instrumental doubling
Number of classes: 21
Classes: ['10_7', '11_7', '12_8', '13_10', '14_9', '15_10', '16_0', '17_11', '18_12', '19_13', '1_1', '20_14', '21_15', '2_1', '3_2', '4_3', '5_4', '6_5', '7_5', '8_6', '9_6']
Number of events in filein:  6243
Last onset at 199.96666666666667
[[0 0 0 ... 0 0 0]
 

{'time': 2.6305601596832275,
 'accuracy (test)': 0.9319455564451561,
 'accuracy (train)': 0.9705646776131358,
 'precision (test)': 0.9578025477707006,
 'precision (train)': 0.9903807615230461,
 'recall (test)': 0.9405785770132916,
 'recall (train)': 0.9737931034482759,
 'f1 (test)': 0.949112426035503,
 'f1 (train)': 0.9820168902136116}

In [10]:
# Single class, with transformations
amo_with_doublings_multiclass(filein,fileout,model=model, tol=0.2, transformations=transformations, multiclass=False, model_path=model_path)


========= PREPROCESSING =========
[PREPROCESSING] Computing and caching results...
{'transformed': 284, 'direct': 41, 'none': 6029}
Original size: (6354, 15)
Dropping [2514, 2517, 2520, 2522, 2525, 2529, 2532, 2535, 2538, 2542, 2552, 2987, 333, 2989, 2991, 2993, 2995, 2997, 2999, 3001, 3003, 3005, 3007, 3009, 3011, 3013, 3015, 3017, 334, 335, 336, 337, 338, 339, 340, 341, 342, 343, 344, 345, 346, 347, 348, 349, 350, 351, 352, 353, 354, 355, 356, 357, 358, 359, 360, 361, 362, 363, 364, 365, 366, 367, 4629, 51, 4631, 53, 4632, 369, 55, 4633, 57, 4634, 59, 4635, 61, 4636, 63, 4637, 65, 4638, 67, 4639, 69, 4640, 71, 4641, 73, 4642, 75, 4643, 77, 4644, 79, 4645, 81, 4646, 549, 83, 4647, 85, 4648, 87, 4649, 89, 4650, 91, 4651, 551, 93, 4652, 552, 553, 95, 4653, 555, 97, 4654, 99, 4655, 101, 4656, 103, 4657, 105, 4658, 383, 107, 4659, 109, 4660, 111, 4661, 113, 4662, 115, 4663, 117, 4664, 119, 4665, 121, 4666, 123, 4667, 125, 4668, 127, 4669, 129, 4670, 131, 4671, 133, 4672, 558, 135, 4673, 

{'time': 2.4707820415496826,
 'accuracy (test)': 0.953579858379229,
 'accuracy (train)': 0.9815069840645289,
 'precision (test)': 0.9554464756199029,
 'precision (train)': 0.9815069840645289,
 'recall (test)': 0.953579858379229,
 'recall (train)': 0.9815069840645289,
 'f1 (test)': 0.9541672949693355,
 'f1 (train)': 0.9815069840645289,
 "time (transpose_{'n_semitones': 12})": 0.44090819358825684,
 "accuracy (test) (transpose_{'n_semitones': 12})": 0.9900497512437811,
 "accuracy (train) (transpose_{'n_semitones': 12})": 0.998133941530168,
 "precision (test) (transpose_{'n_semitones': 12})": 0.9900497512437811,
 "precision (train) (transpose_{'n_semitones': 12})": 0.9981732475299606,
 "recall (test) (transpose_{'n_semitones': 12})": 0.9900497512437811,
 "recall (train) (transpose_{'n_semitones': 12})": 0.998133941530168,
 "f1 (test) (transpose_{'n_semitones': 12})": 0.9900497512437811,
 "f1 (train) (transpose_{'n_semitones': 12})": 0.9981487738002142,
 "time (transpose_{'n_semitones': 24}

In [11]:
# Single class, no transformations
amo_with_doublings_multiclass(filein,fileout,model=model, tol=0.2, transformations=None, multiclass=False, model_path=model_path)



========= PREPROCESSING =========
[PREPROCESSING] Computing and caching results...
Mapping: [[1 'Piccolo' 1 72]
 [2 '2 Flauti' 1 73]
 [3 '2 Oboi' 2 68]
 [4 '2 Clarinetti in A' 3 71]
 [5 '2 Fagotti' 4 70]
 [6 'Corni in F I II' 5 60]
 [7 'Corni in F III IV' 5 60]
 [8 '2 Cornets à piston in A' 6 56]
 [9 '2 Trombe in F' 6 56]
 [10 '2 Tromboni' 7 57]
 [11 'Trombone 3' 7 57]
 [12 'Tuba' 8 58]
 [13 'Timpani' 10 47]
 [14 'Piatti' 9 0]
 [15 'Gran Cassa' 10 47]
 [16 'Arpa' 0 46]
 [17 'Violine I' 11 48]
 [18 'Violine II' 12 48]
 [19 'Viola' 13 48]
 [20 'Violoncelle' 14 45]
 [20 'Violoncelle' 14 48]
 [21 'Contrabasse' 15 45]
 [21 'Contrabasse' 15 48]]

Define covariates and target variable. Target variable encoding
Labels ['10_7' '11_7' '12_8' '13_10' '14_9' '15_10' '16_0' '17_11' '18_12'
 '19_13' '1_1' '20_14' '21_15' '2_1' '3_2' '4_3' '5_4' '6_5' '7_5' '8_6'
 '9_6']
Number of events in filein:  6354
Last onset at 199.96666666666667
y_train, test size: 0.2 , labels: ['10_7' '11_7' '12_8' '13_10'

{'time': 2.4211349487304688,
 'accuracy (test)': 0.953579858379229,
 'accuracy (train)': 0.9815069840645289,
 'precision (test)': 0.9554464756199029,
 'precision (train)': 0.9815069840645289,
 'recall (test)': 0.953579858379229,
 'recall (train)': 0.9815069840645289,
 'f1 (test)': 0.9541672949693355,
 'f1 (train)': 0.9815069840645289}